# Triggering Analysis

Experiment with `I3TruthExtractorPONE` on one random PMT response file per flavor.

In [1]:
import importlib.util
from pathlib import Path

PATHS_PY = Path("/project/def-nahee/kbas/Graphnet-Applications/Metadata/paths.py")

spec = importlib.util.spec_from_file_location("paths", PATHS_PY)
paths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(paths)

pmt_response_paths = paths.STRING340MC_PMT["full_geometry"]
flavors = ["Muon", "Electron", "Tau", "NC"]

for flavor in flavors:
    entry = pmt_response_paths[flavor]
    print(f"{flavor:8s} path={entry['path']} format={entry['format']}")


Muon     path=/home/kbas/scratch/String340MC_pone_offline_version3_plus/Muon_PMT_Response format=gz
Electron path=/home/kbas/scratch/String340MC_pone_offline_version3_plus/Electron_PMT_Response format=gz
Tau      path=/home/kbas/scratch/String340MC_pone_offline_version3_plus/Tau_PMT_Response format=gz
NC       path=/home/kbas/scratch/String340MC_pone_offline_version3_plus/NC_PMT_Response format=gz


## Pick One Random File Per Flavor

In [ ]:
import random

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

def list_pmt_files(flavor):
    entry = pmt_response_paths[flavor]
    pattern = f"*.i3.{entry['format']}"
    return sorted(Path(entry["path"]).rglob(pattern))

selected_files = {}
file_counts = {}

for flavor in flavors:
    files = list_pmt_files(flavor)
    file_counts[flavor] = len(files)
    selected_files[flavor] = rng.choice(files) if files else None

selected_files


## Print Selected Files

In [ ]:
for flavor in flavors:
    print(f"{flavor}: {file_counts[flavor]} files")
    print(f"  selected: {selected_files[flavor]}")


## Truth Extractor Setup

In [ ]:
import sys
import pandas as pd

GRAPHNET_SRC = Path("/project/def-nahee/kbas/graphnet/src")
if str(GRAPHNET_SRC) not in sys.path:
    sys.path.insert(0, str(GRAPHNET_SRC))

from icecube import dataio, icetray
from graphnet.data.extractors.icecube.i3truthextractor import I3TruthExtractorPONE

truth_extractor = I3TruthExtractorPONE(name="truth")
truth_extractor.set_gcd(paths.GCD["340StringMC"])

def extract_truth_dataframe(flavor):
    i3_path = selected_files[flavor]
    if i3_path is None:
        raise FileNotFoundError(f"No PMT response files found for {flavor}")

    rows = []
    i3_file = dataio.I3File(str(i3_path))
    frame_index = 0

    while i3_file.more():
        frame = i3_file.pop_frame()
        if frame.Stop != icetray.I3Frame.DAQ:
            continue

        row = truth_extractor(frame)
        row = {
            "flavor": flavor,
            "source_file": str(i3_path),
            "frame_index": frame_index,
            **row,
        }
        rows.append(row)
        frame_index += 1

    return pd.DataFrame(rows)


## Muon

In [ ]:
muon_df = extract_truth_dataframe("Muon")
print(muon_df.shape)
display(muon_df)


## Electron

In [ ]:
electron_df = extract_truth_dataframe("Electron")
print(electron_df.shape)
display(electron_df)


## Tau

In [ ]:
tau_df = extract_truth_dataframe("Tau")
print(tau_df.shape)
display(tau_df)


## NC

In [ ]:
nc_df = extract_truth_dataframe("NC")
print(nc_df.shape)
display(nc_df)
